# Dashboard ML Result Local Preview

## 1. 코드 전체 요약

이 노트북은 `dash_board/app.py`의 **Overview / ML Result** 화면에서 보던 ML 실험 결과를 로컬 Jupyter 환경에서 재현해 확인하는 용도다. 모델을 다시 학습하거나 예측을 재계산하지 않고, 이미 저장된 대표 실험 목록과 run별 산출물만 읽어서 metric table, learning curve, confusion matrix, feature importance, feature association, score profile을 표시한다.

## 2. 데이터 흐름 요약

- 대표 실험 목록은 `ml/ml_leaderboard_representatives.json`에서 읽는다.
- 각 실험은 `ml/<ml-folder>/ml_outputs/<run-id>/<artifact-prefix>_*` 형식의 산출물로 연결된다.
- `metrics_val.json`과 `train_summary.json`에서 F1, Recall, Precision, AUPRC, threshold, best iteration, learning curve 정보를 가져온다.
- `confusion_matrix_val.csv`, `feature_importance.csv`, `feature_assoc_mixed_val.json` 또는 `feature_assoc_mixed_train.json`이 있으면 추가 시각화에 사용한다.
- 최종 실행 셀에서 `load_dashboard_preview_data()`가 산출물을 모아 `exp_data`를 만들고, `show_dashboard_preview()`가 화면에 표시한다.

## 3. 변경 시 주의점

- 보려는 run만 바꾸려면 다음 셀의 `VIEW_MODE`, `MANUAL_RUNS`, `RUN_DETAIL_LIMIT`만 수정한다.
- artifact 파일명 규칙을 바꾸면 `ARTIFACT_SUFFIXES`, `artifact_prefix()`, `discover_local_run_representatives()`가 함께 영향을 받는다.
- metric key가 바뀌면 `metric_value()`와 `available_learning_curves()`의 매핑을 확인해야 한다.
- 큰 `prediction_scores_val.parquet`는 기본으로 읽지 않는다. 점수 원본 전체를 읽도록 바꾸면 메모리와 실행 시간이 커질 수 있다.
- 누락 artifact는 run 전체 실패가 아니라 섹션별 “파일 없음”으로 표시될 수 있다. 단, `SKIP_INVALID_RUNS=False`이면 일부 누락이 예외로 바뀐다.

## 4. 주석이 추가된 전체 코드

아래 코드 셀 전체에 한국어 주석을 추가했다. 주석은 코드 흐름, 데이터 흐름, 변경 영향, 결과 표시 지점을 중심으로 작성했으며 코드 실행 로직은 변경하지 않는다.

## 5. 확인 필요 사항

- `metrics_val.json` 내부의 `score_profile`, `prediction_scores` 구조는 현재 코드가 기대하는 형태 기준이다. 생성 스크립트가 바뀐 경우 확인 필요.
- `train_summary.json`의 `xgboost_diagnostics.evals_result`와 legacy `learning_curve.curves` 중 어느 쪽이 최신 표준인지 확인 필요.
- `feature_assoc_mixed_val.json`이 없을 때 train association으로 대체 표시하는 것이 현재 리포팅 정책에 맞는지 확인 필요.
- 대표 실험 제외 규칙에서 `ml-00`을 제외하는 것이 최종 대시보드 정책인지 확인 필요.



In [1]:
# 이 셀은 사용자가 직접 바꾸는 설정 영역이다.
# 아래 값들은 데이터 자체를 만들거나 모델을 실행하지 않고, 어떤 저장 산출물을 읽어 화면에 보여줄지만 결정한다.
from __future__ import annotations

# ============================================================
# 사용자 입력값: 보려는 범위가 바뀌면 이 셀만 수정
# ============================================================

# 보기 범위 선택값이다.
# - dashboard_representatives: 리더보드 대표 실험만 읽어 대시보드와 같은 관점으로 본다.
# - all_local_runs: 로컬 ml_outputs 전체를 훑기 때문에 run이 많으면 화면 표시가 길어진다.
# - manual: 아래 MANUAL_RUNS에 지정한 run만 확인할 때 사용한다.
# "dashboard_representatives" | "all_local_runs" | "manual"
VIEW_MODE = "dashboard_representatives"

# manual 모드에서만 사용한다. exp_id는 "ML-03", "ml-03", "ml_03" 모두 허용한다.
MANUAL_RUNS = [
    # {
    #     "exp_id": "ML-03",
    #     "input_run_id": "r00",
    #     "model_run_id": "d00-fixparam",
    #     "description": "ML-03 fixed parameter run",
    # },
]

# 화면 구성 옵션
# False로 바꾸면 해당 섹션을 숨길 뿐, 입력 artifact나 계산 결과를 변경하지 않는다.
SHOW_OVERVIEW = True                  # 대표 실험 모드에서 대시보드 Overview / ML 그래프 표시
SHOW_RUN_COMPARISON = True            # all_local_runs/manual 모드에서 run별 비교 그래프 표시
SHOW_HYPERPARAMETERS = False          # True이면 각 run의 XGBoost parameter table 표시
SHOW_FEATURE_CORRELATION = True       # feature_assoc_mixed_val/train JSON이 있으면 heatmap 표시
SHOW_SCORE_PROFILE = True             # metrics_val.json의 score_profile 표시
SHOW_FILE_MANIFEST = True             # 어떤 artifact가 로드됐는지 요약 표시

# Learning curve는 대시보드 radio 선택지를 노트북에서는 여러 개 동시에 볼 수 있게 둔다.
# metric 이름은 train_summary 안의 XGBoost evals_result key와 dashboard_metric_key() 매핑에 의존한다.
LEARNING_CURVE_METRICS = ["LogLoss", "AUPRC"]

# Feature importance 표시 설정
# TOP_N_FEATURES는 화면에 표시할 개수만 제한한다. feature_importance.csv 원본은 수정하지 않는다.
TOP_N_FEATURES = 20
FEATURE_IMPORTANCE_SORT = "high"      # "high" | "low"

# 많은 run을 볼 때 상세 섹션을 제한하고 싶으면 정수로 설정한다. None이면 전부 표시한다.
RUN_DETAIL_LIMIT = None

# 산출물이 일부 없는 run을 전체 실패로 보지 않고 섹션별로 "파일 없음"을 표시한다.
# 운영 점검에서 누락 artifact를 즉시 실패로 보고 싶으면 False로 바꾼다.
SKIP_INVALID_RUNS = True



In [2]:
# 공통 import와 경로 탐색 셀이다.
# 경로 기준점(BASE_DIR, ML_DIR)을 여기서 확정하므로, 이후 모든 artifact 탐색은 이 값에 의존한다.
import json
import os
import re
from pathlib import Path
from typing import Any, Optional

# matplotlib가 기본 캐시 디렉터리에 쓰지 못하는 환경을 대비해 임시 캐시 경로를 지정한다.
MPLCONFIGDIR = Path("/tmp/matplotlib-codex-cache")
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import pandas as pd
from IPython.display import Markdown, display

# plotly가 있으면 대시보드와 유사한 인터랙티브 차트를 쓰고, 없으면 matplotlib으로 같은 정보를 정적으로 표시한다.
try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    px = None
    go = None
    HAS_PLOTLY = False
    import matplotlib.pyplot as plt
    print("plotly가 없어 matplotlib fallback으로 표시합니다. 대시보드와 같은 인터랙션이 필요하면 `pip install plotly` 후 다시 실행하세요.")


# 현재 노트북 실행 위치가 repo root가 아니어도, ml/과 dash_board/가 있는 상위 디렉터리를 찾아 기준 경로로 쓴다.
def find_repo_root(start: Path) -> Path:
    """현재 위치에서 Graph_AML repository root를 찾는다."""
    resolved = start.resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / "ml").is_dir() and (candidate / "dash_board").is_dir():
            return candidate
    raise FileNotFoundError(f"Cannot find repository root from {start}")


# 이후 모든 상대 경로 계산의 기준이다. 다른 repo에서 실행하면 여기서 잘못된 산출물을 읽는 것을 막는다.
BASE_DIR = find_repo_root(Path.cwd())
ML_DIR = BASE_DIR / "ml"
REPS_PATH = ML_DIR / "ml_leaderboard_representatives.json"

print(f"BASE_DIR = {BASE_DIR}")
print(f"ML_DIR = {ML_DIR}")
print(f"REPS_PATH = {REPS_PATH}")



BASE_DIR = /mnt/d/AML_git/Graph_AML
ML_DIR = /mnt/d/AML_git/Graph_AML/ml
REPS_PATH = /mnt/d/AML_git/Graph_AML/ml/ml_leaderboard_representatives.json


In [3]:
# ============================================================
# Run 선택, artifact 로딩, dashboard-compatible helper
# 이 셀은 "어떤 run을 읽을지"와 "읽은 artifact를 공통 dict 구조로 정리하는 방법"을 정의한다.
# 실제 화면 출력은 뒤쪽 display helper 셀에서 수행한다.
# ============================================================

# VIEW_MODE 허용값이다. 새 모드를 추가하려면 selected_representatives()의 분기와 함께 수정해야 한다.
VALID_VIEW_MODES = {"dashboard_representatives", "all_local_runs", "manual"}
METRIC_COLORS = {"F1": "#4f9cf9", "AUPRC": "#a78bfa", "Recall": "#34d399"}
# run별로 읽을 산출물 파일 suffix 목록이다.
# prefix는 artifact_prefix(rep)가 만들고, suffix는 여기서 관리한다.
# 새 결과 파일을 화면에 연결하려면 이 dict와 load_optional_artifacts()의 로딩 방식을 같이 확장한다.
ARTIFACT_SUFFIXES = {
    "metrics": "metrics_val.json",
    "train_summary": "train_summary.json",
    "confusion_matrix": "confusion_matrix_val.csv",
    "feature_importance": "feature_importance.csv",
    "feature_assoc_val": "feature_assoc_mixed_val.json",
    "feature_assoc_train": "feature_assoc_mixed_train.json",
    "threshold": "threshold.json",
    "feature_columns": "feature_columns.json",
}


# JSON artifact를 읽는 단일 진입점이다. JSON 파싱 실패는 조용히 넘기지 않고 원인 경로를 포함해 예외로 올린다.
def read_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as file:
            return json.load(file)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON: {path}") from exc


# 사용자가 ML-03, ml_03처럼 입력해도 실제 폴더명 ml-03으로 맞춘다.
# 이미 세부 suffix가 붙은 실제 폴더명은 보존한다.
def normalize_ml_folder(value: Any) -> str:
    """Manual input like ML-03/ml_03 is normalized, but real folder suffixes are preserved."""
    text = str(value).strip()
    match = re.fullmatch(r"ml[-_]?([0-9]+)", text, flags=re.IGNORECASE)
    return f"ml-{int(match.group(1)):02d}" if match else text


# 실험 ID는 파일 prefix와 비교하기 쉽도록 ml_03 형태로 표준화한다.
def normalize_experiment_id(value: Any) -> Optional[str]:
    if value is None:
        return None
    text = str(value).strip().replace("-", "_").lower()
    match = re.search(r"ml[_-]?([0-9]+)", text, flags=re.IGNORECASE)
    return f"ml_{int(match.group(1)):02d}" if match else text


def woe_iv_folder_name(ml_folder: str) -> str:
    match = re.match(r"(ml-\d+)", str(ml_folder))
    return match.group(1) if match else str(ml_folder)


# artifact 파일명 앞부분을 만든다. 이 규칙이 바뀌면 모든 산출물 탐색 경로가 같이 바뀐다.
def artifact_prefix(rep: dict[str, Any]) -> str:
    return f"{rep['experiment_id']}__{rep['run_id']}__{rep['model_run_id']}"


# output_dir가 명시된 run은 그 경로를 우선하고, 없으면 ml/<folder>/ml_outputs/<run-id>를 기본 위치로 본다.
def run_output_dir(rep: dict[str, Any]) -> Path:
    if rep.get("output_dir"):
        return Path(rep["output_dir"])
    return ML_DIR / rep["ml_folder"] / "ml_outputs" / rep["run_id"]


def display_label(rep: dict[str, Any]) -> str:
    return f"{woe_iv_folder_name(rep['ml_folder']).upper()} / {rep['run_id']} / {rep['model_run_id']}"


# 대시보드 비교 대상에서 제외할 run을 걸러낸다. ml-00 제외 정책은 확인 필요.
def is_valid_ml_rep(rep: dict[str, Any]) -> bool:
    folder = woe_iv_folder_name(rep.get("ml_folder", ""))
    return bool(re.match(r"ml-\d+", folder)) and folder != "ml-00"


# 대시보드 대표 실험 JSON을 읽어 내부 표준 형태로 정규화한다.
# 여기서 만든 rep dict가 이후 artifact 경로 계산의 입력이 된다.
def load_leaderboard_representatives(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"리더보드 대표 실험 파일 없음: {path}")
    payload = read_json(path)
    if not isinstance(payload, list):
        raise TypeError(f"리더보드 대표 실험 파일은 list여야 합니다: {path}")
    reps: list[dict[str, Any]] = []
    for rep in payload:
        if not isinstance(rep, dict):
            continue
        normalized = dict(rep)
        normalized["ml_folder"] = normalize_ml_folder(normalized.get("ml_folder", ""))
        normalized["experiment_id"] = normalize_experiment_id(normalized.get("experiment_id"))
        reps.append(normalized)
    return reps


# manual 모드 입력을 대시보드 대표 실험과 같은 rep dict 형태로 변환한다.
# 필수값이 빠지면 잘못된 경로를 조용히 만들지 않고 즉시 실패시킨다.
def manual_runs_to_representatives(runs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    reps: list[dict[str, Any]] = []
    for index, run in enumerate(runs, start=1):
        if not isinstance(run, dict):
            raise TypeError(f"MANUAL_RUNS item #{index} must be a dict")
        experiment_id = normalize_experiment_id(run.get("exp_id") or run.get("experiment_id"))
        if experiment_id is None:
            raise ValueError(f"MANUAL_RUNS item #{index}: exp_id 필요")
        run_id = run.get("input_run_id") or run.get("run_id")
        model_run_id = run.get("model_run_id") or run.get("runmodel_id")
        if not run_id or not model_run_id:
            raise ValueError(f"MANUAL_RUNS item #{index}: input_run_id/model_run_id 필요")
        reps.append({
            "ml_folder": normalize_ml_folder(run.get("ml_folder") or experiment_id),
            "experiment_id": experiment_id,
            "run_id": str(run_id),
            "model_run_id": str(model_run_id),
            "description": run.get("description") or run.get("note") or "manual local preview",
            "note": run.get("note", ""),
            "status": run.get("status") or "manual_local_preview",
            "source": "manual",
        })
    return reps


# all_local_runs 모드에서 로컬 ml_outputs를 훑어 metrics 파일명만으로 run 목록을 복원한다.
# 파일명 규칙이 바뀌면 pattern 정규식도 함께 바꿔야 한다.
def discover_local_run_representatives() -> list[dict[str, Any]]:
    """ml_outputs 아래의 *_metrics_val.json 파일명에서 run manifest를 만든다."""
    reps: list[dict[str, Any]] = []
    pattern = re.compile(r"(?P<experiment_id>.+?)__(?P<run_id>.+?)__(?P<model_run_id>.+?)_metrics_val\.json$")
    for metrics_path in sorted(ML_DIR.glob("ml-*/ml_outputs/**/*_metrics_val.json")):
        match = pattern.match(metrics_path.name)
        if not match:
            continue
        rel_parts = metrics_path.relative_to(ML_DIR).parts
        if len(rel_parts) < 4:
            continue
        ml_folder = rel_parts[0]
        groups = match.groupdict()
        reps.append({
            "ml_folder": ml_folder,
            "experiment_id": groups["experiment_id"],
            "run_id": groups["run_id"],
            "model_run_id": groups["model_run_id"],
            "description": "local ml_outputs run",
            "note": "",
            "status": "local_discovered",
            "source": "all_local_runs",
            "output_dir": str(metrics_path.parent),
        })
    return reps


def dedupe_representatives(reps: list[dict[str, Any]]) -> list[dict[str, Any]]:
    deduped: list[dict[str, Any]] = []
    seen: set[str] = set()
    for rep in reps:
        key = f"{rep.get('ml_folder')}::{artifact_prefix(rep)}"
        if key in seen:
            continue
        seen.add(key)
        deduped.append(rep)
    return deduped


# VIEW_MODE에 따라 후보 run을 고르고, 표시 대상과 skip 대상 DataFrame을 분리한다.
# skipped_df는 "왜 화면에 안 나왔는지"를 추적하는 운영 점검용 출력이다.
def selected_representatives() -> tuple[list[dict[str, Any]], pd.DataFrame]:
    if VIEW_MODE not in VALID_VIEW_MODES:
        raise ValueError(f"VIEW_MODE must be one of {sorted(VALID_VIEW_MODES)}: {VIEW_MODE}")

    if VIEW_MODE == "dashboard_representatives":
        candidates = load_leaderboard_representatives(REPS_PATH)
    elif VIEW_MODE == "all_local_runs":
        candidates = discover_local_run_representatives()
    else:
        candidates = manual_runs_to_representatives(MANUAL_RUNS)

    reps: list[dict[str, Any]] = []
    skipped: list[dict[str, Any]] = []
    for rep in dedupe_representatives(candidates):
        if not is_valid_ml_rep(rep):
            skipped.append({**rep, "reason": "invalid_or_ml00_rep"})
            continue
        if rep.get("experiment_id") is None:
            skipped.append({**rep, "reason": "missing_experiment_id"})
            continue
        reps.append(rep)
    return reps, pd.DataFrame(skipped)


def artifact_path(output_dir: Path, prefix: str, suffix: str) -> Path:
    return output_dir / f"{prefix}_{suffix}"


# 한 run에 필요한 artifact를 가능한 만큼 읽어 result dict에 모은다.
# 누락 파일은 missing에 기록하고 계속 진행하므로, 화면에서 파일별 결손을 확인할 수 있다.
def load_optional_artifacts(rep: dict[str, Any]) -> dict[str, Any]:
    prefix = artifact_prefix(rep)
    output_dir = run_output_dir(rep)
    result: dict[str, Any] = {
        "rep": rep,
        "prefix": prefix,
        "output_dir": output_dir,
        "label": display_label(rep),
        "ml": {},
        "files": [],
        "missing": [],
    }

    # artifact별 실제 파일 경로를 만들고 존재 여부를 기록한다.
    # JSON은 dict로, CSV는 DataFrame으로 읽어 뒤쪽 표시 함수가 타입 기준으로 처리한다.
    for key, suffix in ARTIFACT_SUFFIXES.items():
        path = artifact_path(output_dir, prefix, suffix)
        if not path.exists():
            result["missing"].append({"artifact": key, "path": str(path)})
            continue
        result["files"].append({"artifact": key, "path": str(path), "size_mb": path.stat().st_size / 1024 / 1024})
        if key in {"metrics", "train_summary", "threshold", "feature_columns", "feature_assoc_val", "feature_assoc_train"}:
            result["ml"][key] = read_json(path)
        elif key in {"confusion_matrix", "feature_importance"}:
            result["ml"][key] = pd.read_csv(path)

    # feature association은 validation 산출물을 우선한다.
    # val 파일이 없을 때 train 파일로 대체하는 정책은 리포팅 기준에 맞는지 확인 필요.
    if "feature_assoc_val" in result["ml"]:
        result["ml"]["feature_assoc"] = result["ml"]["feature_assoc_val"]
    elif "feature_assoc_train" in result["ml"]:
        result["ml"]["feature_assoc"] = result["ml"]["feature_assoc_train"]

    return result


# 선택된 여러 run의 artifact를 모두 읽어 exp_data로 묶는다.
# exp_data key는 폴더와 prefix를 조합해 중복 run을 구분한다.
def load_dashboard_preview_data() -> tuple[dict[str, dict[str, Any]], pd.DataFrame]:
    reps, skipped_df = selected_representatives()
    exp_data: dict[str, dict[str, Any]] = {}
    skipped_rows = skipped_df.to_dict(orient="records") if not skipped_df.empty else []

    for rep in reps:
        data = load_optional_artifacts(rep)
        if "metrics" not in data["ml"] and "train_summary" not in data["ml"]:
            reason = "metrics/train_summary 산출물이 모두 없음"
            if not SKIP_INVALID_RUNS:
                raise FileNotFoundError(f"{data['label']}: {reason} ({data['output_dir']})")
            skipped_rows.append({**rep, "reason": reason, "output_dir": str(data["output_dir"])})
            continue
        exp_data[f"{rep['ml_folder']}::{data['prefix']}"] = data

    return exp_data, pd.DataFrame(skipped_rows)


# metrics_val.json이 {"metrics": {...}} 형태이거나 metric dict 자체인 경우를 모두 흡수한다.
# 생성 스크립트 버전에 따라 구조가 다를 수 있어 확인 필요 지점이다.
def metrics_payload(metrics_raw: Any) -> dict[str, Any]:
    if not isinstance(metrics_raw, dict):
        return {}
    metrics = metrics_raw.get("metrics", metrics_raw)
    return metrics if isinstance(metrics, dict) else {}


def train_summary_payload(ml: dict[str, Any]) -> dict[str, Any]:
    payload = ml.get("train_summary", {})
    return payload if isinstance(payload, dict) else {}


def infer_model_name(train_summary: dict[str, Any], metrics_raw: dict[str, Any]) -> str:
    run_id = str(train_summary.get("run_id") or metrics_raw.get("run_id") or "").lower()
    if run_id.startswith("xgb"):
        return "XGBoost"
    params = train_summary.get("xgboost_params") or {}
    return "XGBoost" if params else "Unknown"


def eval_keys_for_train_val(evals: dict[str, Any]) -> tuple[Optional[str], Optional[str]]:
    keys = list(evals.keys())
    if "train" in evals and "val" in evals:
        return "train", "val"
    if "validation_0" in evals and "validation_1" in evals:
        return "validation_0", "validation_1"
    if len(keys) >= 2:
        return keys[0], keys[1]
    return None, None


# 학습 곡선은 XGBoost evals_result를 우선 사용하고, 구버전 learning_curve.curves를 보조로 읽는다.
# train/val key 이름이 바뀌면 eval_keys_for_train_val()의 후보를 확장해야 한다.
def available_learning_curves(train_summary: dict[str, Any]) -> tuple[dict[str, tuple[tuple[float, ...], tuple[float, ...]]], str]:
    """대시보드 원천인 xgboost_diagnostics.evals_result를 우선 사용한다."""
    diag = train_summary.get("xgboost_diagnostics") or {}
    evals = diag.get("evals_result") or {}
    source = "xgboost_diagnostics.evals_result"

    if not isinstance(evals, dict) or not evals:
        learning_curve = train_summary.get("learning_curve") or {}
        curves = learning_curve.get("curves") or {}
        if isinstance(curves, dict) and curves.get("train") and curves.get("val"):
            evals = {"train": curves["train"], "val": curves["val"]}
            source = "learning_curve.curves"
        else:
            return {}, source

    train_key, val_key = eval_keys_for_train_val(evals)
    if train_key is None or val_key is None:
        return {}, source

    train_metrics = evals.get(train_key) or {}
    val_metrics = evals.get(val_key) or {}
    available: dict[str, tuple[tuple[float, ...], tuple[float, ...]]] = {}
    for metric, train_vals in train_metrics.items():
        if metric in val_metrics:
            available[str(metric)] = (tuple(train_vals), tuple(val_metrics[metric]))
    return available, source


def dashboard_metric_key(metric_label: str) -> str:
    mapping = {"LogLoss": "logloss", "AUPRC": "aucpr"}
    return mapping.get(metric_label, metric_label)


# 화면에서 쓰는 표시 이름(F1, AUPRC 등)을 artifact 내부 key로 연결한다.
# 새 metric을 추가하면 이 매핑과 figure builder의 metric 목록을 같이 수정한다.
def metric_value(metrics: dict[str, Any], train_summary: dict[str, Any], metric: str) -> Any:
    if metric == "F1":
        return metrics.get("f1")
    if metric == "AUPRC":
        return metrics.get("average_precision") or train_summary.get("best_score")
    if metric == "Recall":
        return metrics.get("recall")
    if metric == "Precision":
        return metrics.get("precision")
    if metric == "Threshold":
        return metrics.get("threshold")
    return None


def sorted_exp_items(exp_data: dict[str, dict[str, Any]]) -> list[tuple[str, dict[str, Any]]]:
    return sorted(
        exp_data.items(),
        key=lambda item: (woe_iv_folder_name(item[1]["rep"]["ml_folder"]), item[1]["rep"].get("run_id", ""), item[1]["rep"].get("model_run_id", "")),
    )


# Overview / ML 선 그래프용 long-form row를 만든다.
# 같은 exp별로 F1, AUPRC, Recall이 metric 컬럼에 쌓이는 구조다.
def overview_metric_rows(exp_data: dict[str, dict[str, Any]]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for _, data in sorted_exp_items(exp_data):
        rep = data["rep"]
        ml = data["ml"]
        metrics = metrics_payload(ml.get("metrics", {}))
        train_summary = train_summary_payload(ml)
        exp = woe_iv_folder_name(rep["ml_folder"]).upper()
        desc = rep.get("description", "") or rep.get("note", "")
        for metric in ("F1", "AUPRC", "Recall"):
            value = metric_value(metrics, train_summary, metric)
            if value is not None:
                rows.append({"exp": exp, "metric": metric, "value": value, "description": desc})
    return rows


def run_comparison_rows(exp_data: dict[str, dict[str, Any]]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for _, data in sorted_exp_items(exp_data):
        rep = data["rep"]
        ml = data["ml"]
        metrics = metrics_payload(ml.get("metrics", {}))
        train_summary = train_summary_payload(ml)
        run_label = display_label(rep)
        desc = rep.get("description", "") or rep.get("note", "")
        for metric in ("F1", "AUPRC", "Recall", "Precision"):
            value = metric_value(metrics, train_summary, metric)
            if value is not None:
                rows.append({"run": run_label, "metric": metric, "value": value, "description": desc})
    return rows


# run별 핵심 지표를 한 행으로 요약한다. 대시보드와 노트북 결과를 대조할 때 첫 번째로 확인하는 표다.
def build_summary_table(exp_data: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for _, data in sorted_exp_items(exp_data):
        rep = data["rep"]
        ml = data["ml"]
        metrics = metrics_payload(ml.get("metrics", {}))
        train_summary = train_summary_payload(ml)
        curves, curve_source = available_learning_curves(train_summary)
        rows.append({
            "label": display_label(rep),
            "prefix": data["prefix"],
            "status": rep.get("status"),
            "F1": metric_value(metrics, train_summary, "F1"),
            "AUPRC": metric_value(metrics, train_summary, "AUPRC"),
            "Recall": metric_value(metrics, train_summary, "Recall"),
            "Precision": metric_value(metrics, train_summary, "Precision"),
            "Threshold": metric_value(metrics, train_summary, "Threshold"),
            "best_iteration": train_summary.get("best_iteration"),
            "feature_count": train_summary.get("feature_count") or metrics.get("feature_count"),
            "curve_metrics": ", ".join(sorted(curves)),
            "curve_source": curve_source if curves else "",
        })
    return pd.DataFrame(rows)


# 어떤 artifact가 실제로 로드됐고 어떤 파일이 빠졌는지 확인하는 manifest 표를 만든다.
def build_file_manifest_table(exp_data: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for _, data in sorted_exp_items(exp_data):
        loaded = {row["artifact"]: row for row in data["files"]}
        for artifact in ARTIFACT_SUFFIXES:
            row = loaded.get(artifact)
            rows.append({
                "label": data["label"],
                "artifact": artifact,
                "loaded": row is not None,
                "size_mb": None if row is None else round(row["size_mb"], 3),
            })
    return pd.DataFrame(rows)



In [4]:
# ============================================================
# Figure builders: dash_board/app.py의 ML Result 의미를 로컬에 맞게 재사용
# 이 셀은 이미 정리된 DataFrame/dict를 그림 객체로 바꾸는 역할만 한다.
# 입력 artifact를 새로 읽거나 저장하지 않는다.
# ============================================================


# 대표 실험별 F1/AUPRC/Recall 흐름을 대시보드 Overview와 같은 관점으로 그린다.
def make_ml_overview_fig(exp_data: dict[str, dict[str, Any]]) -> Any:
    rows = overview_metric_rows(exp_data)
    if not rows:
        raise ValueError("Overview를 그릴 metric row가 없습니다.")
    df = pd.DataFrame(rows)
    exps = df["exp"].unique().tolist()

    if HAS_PLOTLY:
        fig = go.Figure()
        for metric, color in METRIC_COLORS.items():
            sub = df[df["metric"] == metric]
            if sub.empty:
                continue
            dash = "solid" if metric == "F1" else "dash"
            fig.add_trace(go.Scatter(
                x=sub["exp"], y=sub["value"], mode="lines+markers", name=metric,
                line=dict(color=color, width=2, dash=dash), marker=dict(size=8, color=color),
                customdata=sub["description"],
                hovertemplate="<b>%{x}</b><br>" + metric + ": %{y:.4f}<br><i>%{customdata}</i><extra></extra>",
            ))
        desc_map = df.drop_duplicates("exp").set_index("exp")["description"]
        fig.add_trace(go.Scatter(
            x=exps, y=[0] * len(exps), mode="markers", marker=dict(opacity=0, size=16),
            customdata=[desc_map.get(exp, "") for exp in exps],
            hovertemplate="<b>%{x}</b><br><i>%{customdata}</i><extra></extra>", showlegend=False, name="",
        ))
        fig.update_layout(
            title="ML", height=300, margin=dict(t=40, b=20),
            xaxis=dict(categoryorder="array", categoryarray=exps), yaxis=dict(range=[0, 1]),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                        itemclick="toggleothers", itemdoubleclick="toggle"),
        )
        return fig

    fig, ax = plt.subplots(figsize=(10, 4))
    for metric, color in METRIC_COLORS.items():
        sub = df[df["metric"] == metric]
        if sub.empty:
            continue
        ax.plot(sub["exp"], sub["value"], marker="o", label=metric, color=color, linestyle="-" if metric == "F1" else "--")
    ax.set_title("ML")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig


# all_local_runs/manual 모드에서 run 단위 비교를 위한 grouped bar chart를 만든다.
def make_run_comparison_fig(exp_data: dict[str, dict[str, Any]]) -> Any:
    rows = run_comparison_rows(exp_data)
    if not rows:
        raise ValueError("Run comparison을 그릴 metric row가 없습니다.")
    df = pd.DataFrame(rows)
    metric_order = ["F1", "AUPRC", "Recall", "Precision"]

    if HAS_PLOTLY:
        fig = px.bar(
            df, x="run", y="value", color="metric", barmode="group",
            category_orders={"metric": metric_order},
            color_discrete_map={**METRIC_COLORS, "Precision": "#fbbf24"},
            hover_data={"description": True, "value": ":.4f"},
        )
        fig.update_layout(title="Run Comparison", height=max(360, 24 * df["run"].nunique()), margin=dict(t=40, b=120), yaxis=dict(range=[0, 1]))
        fig.update_xaxes(tickangle=-35)
        return fig

    pivot = df.pivot(index="run", columns="metric", values="value")
    ax = pivot[metric_order].plot(kind="bar", figsize=(12, 5))
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", alpha=0.25)
    fig = ax.get_figure()
    fig.tight_layout()
    return fig


# train/val learning curve를 한 그래프에 그리고 best iteration 위치를 세로선으로 표시한다.
# best_iter는 train_summary 기준이므로 early stopping 정책 변경 시 해석을 확인해야 한다.
def make_dashboard_learning_curve_fig(train_vals: tuple[float, ...], val_vals: tuple[float, ...], metric_label: str, best_iter: int) -> Any:
    rows = (
        [{"Iteration": i + 1, "split": "train", metric_label: v} for i, v in enumerate(train_vals)]
        + [{"Iteration": i + 1, "split": "val", metric_label: v} for i, v in enumerate(val_vals)]
    )
    df = pd.DataFrame(rows)
    if HAS_PLOTLY:
        fig = px.line(df, x="Iteration", y=metric_label, color="split", color_discrete_map={"train": "#4f9cf9", "val": "#ff7f0e"})
        fig.add_vline(x=best_iter + 1, line_dash="dash", line_color="#d62728", annotation_text="best", annotation_font_size=11)
        fig.update_layout(height=310, margin=dict(t=40, b=20), legend_title_text="")
        return fig

    fig, ax = plt.subplots(figsize=(10, 4))
    for split, color in [("train", "#4f9cf9"), ("val", "#ff7f0e")]:
        subset = df[df["split"] == split]
        ax.plot(subset["Iteration"], subset[metric_label], label=split, color=color, linewidth=2)
    ax.axvline(best_iter + 1, linestyle="--", color="#d62728", linewidth=1.2, label="best")
    ax.set_xlabel("Iteration")
    ax.set_ylabel(metric_label)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig


# confusion matrix를 AML 관점에서 TP/FN/FP/TN과 비율로 보여준다.
# Recall 해석에서는 FN이 핵심 리스크이므로 row/column 방향을 임의로 바꾸면 안 된다.
def make_cm_fig(tp: int, fn: int, fp: int, tn: int) -> Any:
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (fp + tn) if (fp + tn) > 0 else 0
    z_text = [[f"{tp:,}<br>TPR {tpr:.1%}", f"{fn:,}<br>FNR {fnr:.1%}"], [f"{fp:,}<br>FPR {fpr:.1%}", f"{tn:,}<br>TNR {tnr:.1%}"]]

    if HAS_PLOTLY:
        fig = px.imshow(
            [[tp, fn], [fp, tn]], x=["Pred Fraud", "Pred Normal"], y=["Actual Fraud", "Actual Normal"],
            color_continuous_scale="Blues", text_auto=False, title="",
        )
        fig.update_traces(text=z_text, texttemplate="%{text}", textfont={"size": 11})
        fig.update_coloraxes(showscale=False)
        fig.update_layout(height=290, margin=dict(t=10, b=10))
        return fig

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow([[tp, fn], [fp, tn]], cmap="Blues")
    ax.set_xticks([0, 1], ["Pred Fraud", "Pred Normal"])
    ax.set_yticks([0, 1], ["Actual Fraud", "Actual Normal"])
    for r, row in enumerate(z_text):
        for c, text in enumerate(row):
            ax.text(c, r, text.replace("<br>", "\n"), ha="center", va="center", fontsize=10)
    fig.tight_layout()
    return fig


# XGBoost feature importance 중 gain 기준 상위/하위 피처를 막대로 표시한다.
# gain/weight/cover의 의미는 XGBoost 산출 기준이므로 모델 종류가 바뀌면 확인 필요.
def make_fi_bar_fig(feature_importance: pd.DataFrame, top_n: int, high_first: bool) -> Any:
    if "importance_gain" not in feature_importance.columns:
        raise ValueError("feature_importance.csv에 importance_gain 컬럼이 없습니다.")
    fi_df = (
        feature_importance
        .sort_values("importance_gain", ascending=not high_first)
        .head(min(top_n, len(feature_importance)))
        .sort_values("importance_gain")
        .reset_index(drop=True)
    )
    if "_desc" not in fi_df.columns:
        fi_df["_desc"] = ""

    if HAS_PLOTLY:
        fig = px.bar(
            fi_df, x="importance_gain", y="feature", orientation="h", color="importance_gain", color_continuous_scale="Blues",
            labels={"importance_gain": "Gain", "feature": "Feature"},
            custom_data=[c for c in ["importance_weight", "importance_cover", "rank_by_gain", "_desc"] if c in fi_df.columns],
        )
        fig.update_traces(hovertemplate="<b>%{y}</b><br>Gain: %{x:,.1f}<extra></extra>")
        fig.update_coloraxes(showscale=False)
        fig.update_layout(height=max(400, len(fi_df) * 28), margin=dict(t=40, b=20))
        return fig

    fig, ax = plt.subplots(figsize=(10, max(4, len(fi_df) * 0.3)))
    ax.barh(fi_df["feature"], fi_df["importance_gain"], color="#4f9cf9")
    ax.set_xlabel("Gain")
    ax.grid(True, axis="x", alpha=0.25)
    fig.tight_layout()
    return fig


# gain, weight, cover를 함께 보며 특정 피처가 자주 쓰였는지/큰 이득을 냈는지 구분한다.
def make_fi_scatter_fig(feature_importance: pd.DataFrame, top_n: int, high_first: bool) -> Any:
    needed = {"importance_gain", "importance_weight", "importance_cover"}
    if not needed.issubset(feature_importance.columns):
        missing = sorted(needed - set(feature_importance.columns))
        raise ValueError(f"feature_importance.csv에 필요한 컬럼이 없습니다: {missing}")
    fi_df = (
        feature_importance
        .sort_values("importance_gain", ascending=not high_first)
        .head(min(top_n, len(feature_importance)))
        .reset_index(drop=True)
    )

    if HAS_PLOTLY:
        fig = px.scatter(
            fi_df, x="importance_gain", y="importance_weight", size="importance_cover", hover_name="feature",
            color="importance_gain", color_continuous_scale="Blues",
            labels={"importance_gain": "Gain", "importance_weight": "Weight", "importance_cover": "Cover"},
        )
        fig.update_traces(hovertemplate="<b>%{hovertext}</b><br>Gain: %{x:,.1f}<br>Weight: %{y:,.0f}<extra></extra>")
        fig.update_coloraxes(showscale=False)
        fig.update_layout(height=420, margin=dict(t=20, b=20))
        return fig

    fig, ax = plt.subplots(figsize=(8, 5))
    sizes = (fi_df["importance_cover"] / max(float(fi_df["importance_cover"].max()), 1.0)) * 600
    ax.scatter(fi_df["importance_gain"], fi_df["importance_weight"], s=sizes, alpha=0.6, color="#4f9cf9")
    ax.set_xlabel("Gain")
    ax.set_ylabel("Weight")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    return fig


# feature association matrix를 heatmap으로 표시한다.
# top feature 목록과 association 파일의 feature 이름이 맞지 않으면 표시할 수 없다.
def make_assoc_heatmap(feature_assoc: dict[str, Any], top_features: tuple[str, ...]) -> Any:
    assoc = feature_assoc.get("association", {}) if isinstance(feature_assoc, dict) else {}
    feat_list = feature_assoc.get("features", []) if isinstance(feature_assoc, dict) else []
    matrix = assoc.get("matrix", [])
    metric_matrix = assoc.get("metric_matrix", [])
    if not feat_list or not matrix:
        raise ValueError("Association 데이터 없음")

    all_names = [f.get("name") for f in feat_list]
    if top_features:
        top_set = set(top_features)
        idx = [i for i, name in enumerate(all_names) if name in top_set]
        if not idx:
            raise ValueError("top feature와 association matrix의 feature 이름이 겹치지 않습니다")
        names = [all_names[i] for i in idx]
        matrix = [[matrix[r][c] for c in idx] for r in idx]
        metric_matrix = [[metric_matrix[r][c] for c in idx] for r in idx] if metric_matrix else []
    else:
        names = all_names

    if HAS_PLOTLY:
        custom = [[[metric_matrix[r][c] if metric_matrix else ""] for c in range(len(names))] for r in range(len(names))]
        fig = go.Figure(go.Heatmap(
            z=matrix, x=names, y=names, colorscale="RdBu_r", zmin=-1, zmax=1, customdata=custom,
            hovertemplate="<b>X: %{x}</b><br><b>Y: %{y}</b><br><b>%{customdata[0]}: %{z:.4f}</b><extra></extra>", showscale=False,
        ))
        fig.update_layout(height=max(400, len(names) * 28), margin=dict(t=40, b=20), yaxis=dict(scaleanchor="x", tickfont_size=9), xaxis=dict(showticklabels=False, ticks=""))
        return fig

    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(matrix, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_yticks(range(len(names)), names, fontsize=8)
    ax.set_xticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    return fig



In [5]:
# ============================================================
# Display helpers
# 이 셀은 앞에서 만든 loader와 figure builder를 조합해 노트북 화면을 구성한다.
# 결과 저장은 하지 않고 display()만 수행한다.
# ============================================================


# 한 run의 핵심 metric, row count, threshold, best iteration, feature count를 표로 보여준다.
# metric 값은 metrics_val.json을 우선 쓰고 일부 값은 train_summary에서 보완한다.
def display_metric_table(data: dict[str, Any]) -> None:
    ml = data["ml"]
    metrics_raw = ml.get("metrics", {})
    metrics = metrics_payload(metrics_raw)
    train_summary = train_summary_payload(ml)
    model_name = infer_model_name(train_summary, metrics_raw if isinstance(metrics_raw, dict) else {})
    metrics_raw_dict = metrics_raw if isinstance(metrics_raw, dict) else {}
    val_label_summary = train_summary.get("val_label_summary") or metrics_raw_dict.get("label_summary", {})
    val_rows = train_summary.get("val_rows") or metrics_raw_dict.get("data_profile", {}).get("val_rows")
    feature_count = train_summary.get("feature_count") or metrics_raw_dict.get("feature_count")

    display(pd.DataFrame([{
        "model": model_name,
        "F1": metric_value(metrics, train_summary, "F1"),
        "AUPRC": metric_value(metrics, train_summary, "AUPRC"),
        "Recall": metric_value(metrics, train_summary, "Recall"),
        "Precision": metric_value(metrics, train_summary, "Precision"),
        "Threshold": metric_value(metrics, train_summary, "Threshold"),
        "Train rows": train_summary.get("train_rows"),
        "Val rows": val_rows,
        "Val positive rate": val_label_summary.get("positive_ratio"),
        "Best iteration": train_summary.get("best_iteration"),
        "Training sec": train_summary.get("training_time_sec"),
        "Feature count": feature_count,
    }]))


# SHOW_HYPERPARAMETERS=True일 때 XGBoost parameter를 표시한다.
# 파라미터는 해석용 출력이며 여기서 모델을 재생성하지 않는다.
def display_hyperparameters(train_summary: dict[str, Any]) -> None:
    params = train_summary.get("xgboost_params") or {}
    if not params:
        display(Markdown("Hyper Parameters: 파일 없음 또는 값 없음"))
        return
    display(pd.DataFrame([{"parameter": key, "value": value} for key, value in params.items() if value is not None]))


# confusion_matrix_val.csv가 있으면 우선 사용하고, 없으면 metrics JSON 내부 값을 찾아 대체한다.
def confusion_counts(ml: dict[str, Any]) -> Optional[dict[str, int]]:
    conf_mat = ml.get("confusion_matrix")
    if isinstance(conf_mat, pd.DataFrame) and not conf_mat.empty:
        row = conf_mat.iloc[0]
        return {name: int(row.get(name, 0)) for name in ("tn", "fp", "fn", "tp")}
    metrics_raw = ml.get("metrics", {})
    metrics = metrics_payload(metrics_raw)
    if all(name in metrics for name in ("tn", "fp", "fn", "tp")):
        return {name: int(metrics.get(name, 0)) for name in ("tn", "fp", "fn", "tp")}
    conf = metrics_raw.get("confusion_matrix", {}) if isinstance(metrics_raw, dict) else {}
    if all(name in conf for name in ("tn", "fp", "fn", "tp")):
        return {name: int(conf.get(name, 0)) for name in ("tn", "fp", "fn", "tp")}
    return None


# train_summary에서 learning curve를 꺼내 LEARNING_CURVE_METRICS에 지정된 지표만 표시한다.
def display_learning_curves(data: dict[str, Any]) -> None:
    train_summary = train_summary_payload(data["ml"])
    curves, source = available_learning_curves(train_summary)
    if not curves:
        display(Markdown("학습 곡선 데이터 없음: `train_summary.xgboost_diagnostics.evals_result` 확인 필요"))
        return
    display(Markdown(f"curve source: `{source}`"))
    for metric_label in LEARNING_CURVE_METRICS:
        metric_key = dashboard_metric_key(metric_label)
        if metric_key not in curves:
            display(Markdown(f"- `{metric_label}` curve 없음. available={sorted(curves)}"))
            continue
        train_vals, val_vals = curves[metric_key]
        best_iter = int(train_summary.get("best_iteration") or 0)
        fig = make_dashboard_learning_curve_fig(train_vals, val_vals, metric_label, best_iter)
        if HAS_PLOTLY:
            fig.update_layout(title=f"{data['label']} {metric_label} Learning Curve")
        else:
            fig.axes[0].set_title(f"{data['label']} {metric_label} Learning Curve")
        display(fig)


# feature_importance.csv를 읽은 결과가 있으면 중요도 그래프를 그리고, 상위 feature 이름을 correlation heatmap 입력으로 넘긴다.
def display_feature_importance(data: dict[str, Any]) -> tuple[str, ...]:
    feat_imp = data["ml"].get("feature_importance")
    if not isinstance(feat_imp, pd.DataFrame) or feat_imp.empty:
        display(Markdown("Feature importance 파일 없음"))
        return ()
    high_first = FEATURE_IMPORTANCE_SORT.lower() != "low"
    top_names = tuple(
        feat_imp.sort_values("importance_gain", ascending=not high_first)
        .head(min(TOP_N_FEATURES, len(feat_imp)))["feature"]
        .tolist()
    )
    display(make_fi_bar_fig(feat_imp, TOP_N_FEATURES, high_first))
    display(Markdown("##### Weight vs Gain"))
    try:
        display(make_fi_scatter_fig(feat_imp, TOP_N_FEATURES, high_first))
        display(Markdown("버블 크기 = Cover"))
    except ValueError as exc:
        display(Markdown(f"Weight vs Gain 표시 불가: `{exc}`"))
    return top_names


# feature importance 상위 피처만 골라 association heatmap을 좁혀 보여준다.
# association 계산 방식은 artifact 내부 metadata를 표시하되, 산출 기준은 생성 스크립트에서 확인 필요.
def display_feature_correlation(data: dict[str, Any], top_feature_names: tuple[str, ...]) -> None:
    feature_assoc = data["ml"].get("feature_assoc")
    if not SHOW_FEATURE_CORRELATION:
        display(Markdown("Feature Correlation: 설정에서 숨김"))
        return
    if not isinstance(feature_assoc, dict):
        display(Markdown("feature_assoc_mixed 파일 없음"))
        return
    split_label = feature_assoc.get("split", "?")
    methods = feature_assoc.get("association_methods", {})
    display(Markdown(
        f"Split: `{split_label}` | numeric-numeric: `{methods.get('numeric_numeric', '?')}` | "
        f"cat-cat: `{methods.get('categorical_categorical', '?')}` | mixed: `{methods.get('numeric_categorical', '?')}`"
    ))
    try:
        display(make_assoc_heatmap(feature_assoc, top_feature_names))
    except ValueError as exc:
        display(Markdown(f"Association 데이터 없음: `{exc}`"))


# prediction score 원본 parquet를 직접 읽지 않고 metrics_val.json에 저장된 분위수/메타데이터만 표시한다.
# 전체 점수 분포를 상세 분석하려면 별도 셀에서 parquet 로딩 비용을 확인해야 한다.
def display_score_profile(data: dict[str, Any]) -> None:
    if not SHOW_SCORE_PROFILE:
        display(Markdown("Score Profile: 설정에서 숨김"))
        return
    metrics_raw = data["ml"].get("metrics", {})
    if not isinstance(metrics_raw, dict):
        display(Markdown("Score profile 없음"))
        return
    score_profile = metrics_raw.get("score_profile") or {}
    prediction_scores = metrics_raw.get("prediction_scores") or {}
    rows = []
    for label, values in [
        ("all", score_profile.get("probability_quantiles") or {}),
        ("positive", score_profile.get("positive_score_quantiles") or {}),
        ("negative", score_profile.get("negative_score_quantiles") or {}),
    ]:
        if values:
            row = {"group": label}
            row.update(values)
            rows.append(row)
    if rows:
        display(pd.DataFrame(rows))
    else:
        display(Markdown("Score quantile profile 없음"))

    meta_rows = []
    for key in ["predicted_positive_count", "predicted_positive_rate"]:
        if key in score_profile:
            meta_rows.append({"item": key, "value": score_profile[key]})
    for key in ["file_name", "rows", "score_column", "label_column", "split", "sampled"]:
        if key in prediction_scores:
            meta_rows.append({"item": f"prediction_scores.{key}", "value": prediction_scores[key]})
    if meta_rows:
        display(pd.DataFrame(meta_rows))
    if prediction_scores.get("path"):
        display(Markdown(f"prediction score parquet는 기본 미로드: `{prediction_scores.get('path')}`"))


# 한 run에 대한 상세 섹션을 순서대로 출력한다.
# 이 함수의 출력 순서가 노트북 사용자가 결과를 따라가는 기본 흐름이다.
def show_ml_result_section(data: dict[str, Any]) -> None:
    rep = data["rep"]
    display(Markdown(f"## {data['label']}"))
    if rep.get("description"):
        display(Markdown(f"**Description**: {rep['description']}"))
    if rep.get("note"):
        display(Markdown(f"**Note**: {rep['note']}"))
    display(pd.DataFrame([{"prefix": data["prefix"], "output_dir": str(data["output_dir"])}]))

    display(Markdown("### Metrics"))
    display_metric_table(data)

    if SHOW_HYPERPARAMETERS:
        display(Markdown("### Hyper Parameters"))
        display_hyperparameters(train_summary_payload(data["ml"]))

    display(Markdown("### Learning Curve"))
    display_learning_curves(data)

    display(Markdown("### Confusion Matrix"))
    counts = confusion_counts(data["ml"])
    if counts is None:
        display(Markdown("Confusion matrix 파일 없음"))
    else:
        display(make_cm_fig(tp=counts["tp"], fn=counts["fn"], fp=counts["fp"], tn=counts["tn"]))
        display(pd.DataFrame([counts]))

    display(Markdown("### Feature Importance"))
    top_feature_names = display_feature_importance(data)

    display(Markdown("### Feature Correlation"))
    display_feature_correlation(data, top_feature_names)

    display(Markdown("### Score Profile"))
    display_score_profile(data)


# 전체 노트북 화면의 최상위 표시 함수다.
# Summary -> skipped/artifact manifest -> overview/run comparison -> run별 상세 순서로 보여준다.
def show_dashboard_preview(exp_data: dict[str, dict[str, Any]], skipped_df: pd.DataFrame) -> None:
    display(Markdown("# Local Dashboard Preview"))
    display(Markdown(f"VIEW_MODE = `{VIEW_MODE}` | loaded runs = `{len(exp_data)}`"))

    display(Markdown("## Summary"))
    summary_df = build_summary_table(exp_data)
    if summary_df.empty:
        display(Markdown("로드된 실험이 없습니다."))
    else:
        display(summary_df)

    if not skipped_df.empty:
        display(Markdown("## Skipped Runs"))
        display(skipped_df)

    if SHOW_FILE_MANIFEST and exp_data:
        display(Markdown("## Artifact Manifest"))
        display(build_file_manifest_table(exp_data))

    if VIEW_MODE == "dashboard_representatives" and SHOW_OVERVIEW and exp_data:
        display(Markdown("## Overview / ML"))
        display(make_ml_overview_fig(exp_data))
    elif SHOW_RUN_COMPARISON and exp_data:
        display(Markdown("## Run Comparison"))
        display(make_run_comparison_fig(exp_data))

    display(Markdown("# ML Result"))
    items = sorted_exp_items(exp_data)
    if RUN_DETAIL_LIMIT is not None:
        items = items[: int(RUN_DETAIL_LIMIT)]
    for _, data in items:
        show_ml_result_section(data)



In [6]:
# 최종 실행 지점이다.
# exp_data에는 run별 로드된 artifact와 표시용 metadata가 들어가고,
# skipped_df에는 필터링되었거나 필수 artifact가 없어 제외된 run과 사유가 들어간다.
exp_data, skipped_df = load_dashboard_preview_data()
show_dashboard_preview(exp_data, skipped_df)



# Local Dashboard Preview

VIEW_MODE = `dashboard_representatives` | loaded runs = `5`

## Summary

,label,prefix,status,F1,AUPRC,Recall,Precision,Threshold,best_iteration,feature_count,curve_metrics,curve_source
0,ML-01 / r02 / d00-fixparam-format-out,ml_01__r02__d00-fixparam-format-out,candidate_val_only,0.155804,0.070340,0.231764,0.117345,0.964972,299,72,"aucpr, logloss",xgboost_diagnostics.evals_result
1,ML-02 / r02 / d00-optuna-out-format-rev2-t50,ml_02__r02__d00-optuna-out-format-rev2-t50,candidate_val_only,0.295983,0.213432,0.258541,0.346106,0.613622,1998,100,"aucpr, logloss",xgboost_diagnostics.evals_result
2,ML-03 / r00 / d00-fixparam,ml_03__r00__d00-fixparam,candidate_val_only,0.176956,0.089932,0.247461,0.137718,0.946181,297,146,"aucpr, logloss",xgboost_diagnostics.evals_result
3,ML-04 / r01 / d00-fixparam,ml_04__r01__d00-fixparam,candidate_val_only,0.085819,0.057680,0.193906,0.055104,0.607146,4,188,"aucpr, logloss",xgboost_diagnostics.evals_result
4,ML-05 / r00 / d00-optuna_t25,ml_05__r00__d00-optuna_t25,candidate_val_only,0.470090,0.445531,0.409972,0.550868,0.377790,1593,236,"aucpr, logloss",xgboost_diagnostics.evals_result


## Skipped Runs

...

Split: `val` | numeric-numeric: `pearson` | cat-cat: `cramers_v` | mixed: `correlation_ratio_eta`

### Score Profile

,group,q00,q01,q05,q10,q25,q50,q75,q90,q95,q99,q100
0,all,2.587149e-14,9.562314e-09,3.795422e-08,7.942571e-08,2.819977e-07,0.000001,0.000008,0.000082,0.000327,0.004185,0.999999
1,positive,1.353154e-06,5.443787e-06,4.492802e-05,1.485197e-04,2.388114e-03,0.146547,0.885346,0.994058,0.999213,0.999955,0.999999
2,negative,2.587149e-14,9.551392e-09,3.791321e-08,7.931029e-08,2.814608e-07,0.000001,0.000008,0.000081,0.000317,0.003706,0.999677


,item,value
0,predicted_positive_count,806
1,predicted_positive_rate,0.000794
2,prediction_scores.file_name,ml_05__r00__d00-optuna_t25_prediction_scores_v...
3,prediction_scores.rows,1015633
4,prediction_scores.score_column,score
5,prediction_scores.label_column,label
6,prediction_scores.split,val
7,prediction_scores.sampled,False


prediction score parquet는 기본 미로드: `/mnt/d/AML_git/Graph_AML/ml/ml-05/ml_outputs/r00/ml_05__r00__d00-optuna_t25_prediction_scores_val.parquet`